In [1]:
import re
import json
import time
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.chrome.options import Options
from selenium.common.exceptions import StaleElementReferenceException


In [2]:
def login(driver, data):
    driver.get("https://accounts.google.com/signin")

    # === Enter Email ===
    time.sleep(10)
    email_input = driver.find_element(By.XPATH, "//input[@type='email']")
    email_input.send_keys(data["email"])
    email_input.send_keys(Keys.RETURN)

    # === Enter Password ===
    time.sleep(10)
    password_input = driver.find_element(By.XPATH, "//input[@type='password']")
    password_input.send_keys(data["password"])
    password_input.send_keys(Keys.RETURN)
    

In [3]:
# def fill(driver):
#     form_data = {
#         "Name": "Taha Ahmed",
#         "Email": "taha.ahmed@example.com",
#         "Contact No.": "1234567890",
#         "Comments": "This is a test comment"
#     }
#     # === Go to Google Form ===
#     time.sleep(10)
#     driver.get("https://docs.google.com/forms/d/e/1FAIpQLSdiDl_sgvqMFtOv45fUCVegya6IbqattwhpgFAe9im3PmD_Xw/viewform")
#     WebDriverWait(driver, 10).until(
#             EC.presence_of_element_located((By.CLASS_NAME, "geS5n"))
#         )
        
#         # Find all question containers with class 'geS5n'
#     question_containers = driver.find_elements(By.CLASS_NAME, "geS5n")
    
#     for container in question_containers:
#         # Extract the label (first text in the question title)
#         try:
#             label = container.text.strip()
#             label = label.replace("*", "").strip()
#             first_line = label.split('\n')[0]  # Get only the first line
#             print(container)
#             print(first_line)
#             input_field = None
#             try:
#                 input_field = container.find_element(By.TAG_NAME, "input")
#             except:
#                 try:
#                     input_field = container.find_element(By.TAG_NAME, "textarea")
#                 except:
#                     continue  # Skip if no input or textarea is found

#             # Match the label with form_data keys and fill the field
#             for key in form_data:
#                 if key.lower() in label.lower():  # Case-insensitive matching
#                     input_field.send_keys(form_data[key])
#                     print(f"Filled '{key}' with '{form_data[key]}'")
#                     break  # Exit loop once the field is filled
#         except Exception as e:
#             print(f"Error processing field with label '{label}': {e}")
#             continue

#     time.sleep(1)


#     # Close the browser
#     # driver.quit()
    

In [4]:
def fill(driver, data):
    """
    Fills out a Google Form using a generalized approach with regex to skip placeholders.

    This function finds all <input> and <textarea> fields, traverses up the DOM to find
    their labels, and uses a regular expression to ignore common instructional text
    like "Your answer", making the label detection more accurate and robust.
    """

        
    filled_fields = set() # To track which labels have already been filled
    time.sleep(10)
    # === Go to Google Form ===
    driver.get("https://docs.google.com/forms/d/e/1FAIpQLSdiDl_sgvqMFtOv45fUCVegya6IbqattwhpgFAe9im3PmD_Xw/viewform")
    
    WebDriverWait(driver, 10).until(
        EC.presence_of_element_located((By.TAG_NAME, "input"))
    )
    
    # Regex to identify and skip common placeholder/instructional text.
    # It's case-insensitive and matches phrases like "Your answer", "Enter your text", etc.
    placeholder_regex = re.compile(
        r'^(your|enter|my|the) (answer|response|text|name|phone number|comment)', 
        re.IGNORECASE
    )

    # Regex patterns for matching form labels
    regex_patterns = {
        "email": r'\b(?:e-?mail|email(?: address)?)\b',
        "password": r'\b(password|pass|secret)\b',
        "full_name": r'\b(?:full|complete|legal) name\b',
        "first_name": r'\b(?:first|given|fore) name\b',
        "last_name": r'\b(?:last|family|sur)name\b',
        "roll_number": r'\b(?:roll(?: no\.?| number)?|student (?:id|number)|registration (?:no\.?|number))\b',
        "cgpa": r'\b(?:cgpa|cumulative gpa|grade point average|gpa)\b',
        "phone": r'\b(?:phone(?: no\.?| number)?|contact(?: no\.?| number)?|mobile|cell)\b',
        "address": r'\b(?:address|street(?: address)?|mailing address)\b',
        "city": r'\b(city)\b',
        "state": r'\b(?:state|province|region)\b',
        "zip_code": r'\b(?:zip|postal|post) code\b',
        "country": r'\b(country|nation)\b',
        "date_of_birth": r'\b(?:date of birth|dob|birthdate)\b',
        "nationality": r'\b(nationality)\b',
        "linkedin": r'\b(?:linkedin(?: url| profile)?)\b',
        "github": r'\b(?:github(?: url| profile)?|git)\b',
        "portfolio": r'\b(?:portfolio|website|url)\b',
        "university": r'\b(?:university|college|institution|school)\b',
        "resume": r'\b(?:resume|cv|curriculum vitae)\b',
        "cover_letter": r'\b(?:cover letter|covering letter)\b',
        "college": r'\b(?:college|university|institution|school)\b',
        "degree": r'\b(?:degree|education|qualification)\b',
        "major": r'\b(?:major|field of study|specialization)\b',
        "minor": r'\b(minor)\b',
        "graduation_year": r'\b(?:graduat(?:ion|ed)(?: year)?|year of graduation|completion year)\b',
        "expected_graduation": r'\b(?:expected graduation(?: date| year)?|grad year)\b',
        "relevant_coursework": r'\b(?:relevant )?coursework\b',
        "current_position": r'\b(?:current|present) (?:position|job title|role)\b',
        "current_company": r'\b(?:current|present) (?:company|employer)\b',
        "years_experience": r'\b(?:years of )?(?:experience|exp)\b',
        "total_experience": r'\b(?:total|overall) (?:experience|exp)\b',
        "previous_company": r'\b(?:previous|past) (?:company|employer)\b',
        "previous_position": r'\b(?:previous|past) (?:position|job title|role)\b',
        "internship_experience": r'\b(?:internship|co-op) experience\b',
        "work_history": r'\b(?:work|employment) history|experience\b',
        "programming_languages": r'\b(?:programming |technical )?languages|skills\b',
        "frameworks": r'\b(frameworks|libraries)\b',
        "databases": r'\b(databases)\b',
        "tools": r'\b(tools|technologies)\b',
        "soft_skills": r'\b(?:soft|interpersonal) skills\b',
        "certifications": r'\b(certifications|licenses|credentials)\b',
        "desired_salary": r'\b(?:desired|expected) salary|salary expectations?\b',
        "salary_range_min": r'\b(?:salary|pay) (?:range|scale) min(?:imum)?\b',
        "salary_range_max": r'\b(?:salary|pay) (?:range|scale) max(?:imum)?\b',
        "work_authorization": r'\b(?:work authorization|employment eligibility|visa status)\b',
        "visa_sponsorship": r'\b(?:visa|sponsorship) (?:sponsorship|required|needed)\b',
        "willing_to_relocate": r'\b(?:willing to )?relocate\b',
        "remote_work": r'\b(?:remote|wfh|work from home)\b',
        "start_date": r'\b(?:available|earliest) start date|availability\b',
        "availability": r'\b(availability|available to start)\b',
        "notice_period": r'\bnotice period\b'
    }

    # Find all input and textarea elements
    input_fields = driver.find_elements(By.TAG_NAME, "input")
    textarea_fields = driver.find_elements(By.TAG_NAME, "textarea")
    all_fields = input_fields + textarea_fields

    for field in all_fields:
        # Skip buttons and other non-text input types
        try:
            if field.get_attribute('type') not in ['text', 'email', 'tel', 'url', None]: # 'None' for textareas
                 continue
        except StaleElementReferenceException:
            continue

        parent_element = field
        try:
            # Traverse up a maximum of 5 parent levels to find the question label
            for _ in range(10):
                parent_element = parent_element.find_element(By.XPATH, "..")
                raw_text = parent_element.text.strip()
                print(f"Raw text from parent: '{raw_text}'", "Element " + str(parent_element))
                
                if raw_text:
                    # The question is typically the first line of text
                    potential_label = raw_text.split('\n')[0].replace("*", "").strip()

                    # Use regex to check if the found text is a placeholder
                    if placeholder_regex.match(potential_label):
                        print(f"Skipping placeholder text: '{potential_label}'")
                        continue  # Skip this text and check the next parent up

                    # Check if this label matches a key in our regex patterns
                    for key, pattern in regex_patterns.items():
                        if re.search(pattern, potential_label, re.IGNORECASE) and key in data and key not in filled_fields:
                            field.send_keys(data[key])
                            print(f"Filled '{potential_label}' with '{data[key]}' for key '{key}'")
                            filled_fields.add(key)  # Mark as filled
                            break  # Exit inner loop and move to the next field
                    else:
                        continue  # No match found, check next parent
                    break  # Match found and filled, exit parent traversal
        except StaleElementReferenceException:
            print("Encountered a stale element, skipping.")
            continue
        except Exception as e:
            # Silently continue if an error occurs while processing a field
            # print(f"Could not process a field. Error: {e}")
            continue
                
    time.sleep(1)

    # Close the browser
    # driver.quit()

In [5]:
if __name__ == '__main__':
    option = webdriver.ChromeOptions()
    option.add_argument("--incognito")
    # option.add_argument("--headless")

    # Load form data from user_config.json
    with open('user_config.json', 'r') as f:
        data = json.load(f)

    browser = webdriver.Chrome(options = option)
    login(browser, data)
    fill(browser, data)

NoSuchDriverException: Message: Unable to obtain driver for chrome; For documentation on this error, please visit: https://www.selenium.dev/documentation/webdriver/troubleshooting/errors/driver_location
